# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneha27patel/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd

url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

features = ['impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update']
X = df[features].fillna(0)
df['target'] = (df['trend_direction'] == 'down') & (df['avg_position'] <= 20)
y = df['target']

print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
print("\nFeature sample:")
print(X.head())

Feature matrix shape: (30000, 5)
Target distribution:
target
False    18248
True     11752
Name: count, dtype: int64

Feature sample:
   impressions_90d  avg_position   ctr  content_age_days  \
0             3803          10.6  0.76               187   
1            15320          20.3  0.05               445   
2            12581          36.5  0.09               141   
3            11751           6.2  0.49               463   
4            19140          44.0  0.13               263   

   days_since_last_update  
0                      20  
1                      25  
2                      20  
3                      22  
4                      14  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# Feature Notes:
# impressions_90d: total search impressions over 90 days. No missing values. Available before decision.
# avg_position: average Google search position. No missing values. Available before decision.
# ctr: click-through rate percentage. No missing values. Available before decision.
# content_age_days: days since content was created. No missing values. Available before decision.
# days_since_last_update: days since last edit. No missing values. Available before decision.
print("All 5 features are observable signals available BEFORE the prediction moment.")

All 5 features are observable signals available BEFORE the prediction moment.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# CLEAN model (no leakage)
clf_clean = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_clean.fit(X, y)
preds_clean = clf_clean.predict(X)
prec_clean = precision_score(y, preds_clean)
print(f"CLEAN model Precision: {prec_clean:.4f}")

# LEAKED model (adding trend_pct which leaks the target)
X_leaked = X.copy()
X_leaked['trend_pct'] = df['trend_pct'].fillna(0)
clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaked.fit(X_leaked, y)
preds_leaked = clf_leaked.predict(X_leaked)
prec_leaked = precision_score(y, preds_leaked)
print(f"LEAKED model Precision: {prec_leaked:.4f}")

print(f"\nLeakage detected! Score jumped from {prec_clean:.4f} to {prec_leaked:.4f}")
print("trend_pct leaks the target because trend_direction is derived from it.")


CLEAN model Precision: 0.6074
LEAKED model Precision: 0.9999

Leakage detected! Score jumped from 0.6074 to 0.9999
trend_pct leaks the target because trend_direction is derived from it.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
# Completed - no additional leakage found in remaining features.
print("Leakage audit complete.")


Leakage audit complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.